# Ejercicio 5

In [ ]:
import simpy
import random
import numpy as np

# Parámetros
np.random.seed(1)
random.seed(1)

LAMBDA = 1 / 5

MU = {
    1: 1 / 2,
    2: 1 / 4,
    3: 1 / 4,
    4: 1 / 3,
    5: 1 / 2,
    6: 1 / 3
}

N_CLIENTES = 100000

# Crear entorno simpy
env = simpy.Environment()

chatbots = {
    i: simpy.Resource(env, capacity=1)
    for i in range(1, 7)
}

# Métricas
esperas = {i: [] for i in range(1, 7)}
tiempos_nodo = {i: [] for i in range(1, 7)}
ocupacion = {i: 0 for i in range(1, 7)}
visitas_nodo = {i: 0 for i in range(1, 7)}

tiempo_sistema = []


# Tránsito
def siguiente_nodo(actual):
    if actual == 1:
        return np.random.choice([2, 3, 6], p=[0.5, 0.3, 0.2])
    if actual == 2:
        return np.random.choice([3, 4], p=[0.6, 0.4])
    if actual == 3:
        return 4
    if actual == 4:
        return np.random.choice([5, 6], p=[0.9, 0.1])
    if actual == 5:
        return np.random.choice([1, 7], p=[0.1, 0.9])
    if actual == 6:
        return np.random.choice([2, 3, 4, 5, 7], p=[0.2, 0.2, 0.2, 0.2, 0.2])
    return 7


# Cliente
def cliente(env, nombre):
    llegada = env.now
    nodo = np.random.choice([1, 3, 4], p=[0.6, 0.1, 0.3])
    visitas = 0

    while nodo != 7:

        visitas += 1
        visitas_nodo[nodo] += 1

        recurso = chatbots[nodo]

        instante_cola = env.now

        with recurso.request() as req:

            yield req

            espera = env.now - instante_cola
            esperas[nodo].append(espera)

            servicio = random.expovariate(MU[nodo])

            yield env.timeout(servicio)

            fin_servicio = env.now

            tiempos_nodo[nodo].append(fin_servicio - instante_cola)
            ocupacion[nodo] += servicio

        nodo = siguiente_nodo(nodo)

    tiempo_sistema.append(env.now - llegada)

# Llegadas
def llegadas(env):

    for i in range(N_CLIENTES):

        env.process(cliente(env, f"C{i}"))

        interarrival = random.expovariate(LAMBDA)
        yield env.timeout(interarrival)

# Ejecutar
env.process(llegadas(env))
env.run()

# Resultados
print("\nTiempo medio de espera")

for i in range(1, 7):
    print(
        f"Chatbot {i}: "
        f"{np.mean(esperas[i]):.3f}"
    )

print("\nUtilización")

tiempo_total = env.now

for i in range(1, 7):

    rho = ocupacion[i] / tiempo_total

    print(
        f"Chatbot {i}: {rho:.3f}"
    )

print("\nNúmero medio en el sistema")


for i in range(1, 7):
    N = (visitas_nodo[i] / tiempo_total)* np.mean(tiempos_nodo[i])

    print(f"Chatbot {i}: {N:.3f}")

print("\nTiempo medio en sistema")

print(np.mean(tiempo_sistema))


Tiempo medio de espera
Chatbot 1: 0.805
Chatbot 2: 1.973
Chatbot 3: 3.804
Chatbot 4: 6.145
Chatbot 5: 1.450
Chatbot 6: 0.528

Utilización
Chatbot 1: 0.286
Chatbot 2: 0.324
Chatbot 3: 0.485
Chatbot 4: 0.673
Chatbot 5: 0.424
Chatbot 6: 0.151

Número medio en el sistema
Chatbot 1: 0.400
Chatbot 2: 0.484
Chatbot 3: 0.946
Chatbot 4: 2.049
Chatbot 5: 0.732
Chatbot 6: 0.178

Tiempo medio en sistema
23.9030639499074
